# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("/content/drive/MyDrive/final_dl")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 2. Load and Preprocess Data

In [4]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [5]:
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

train: 3109
val: 1048
test: 1008


In [6]:
from transformers import AutoProcessor, AutoModelForVision2Seq

CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Reply with only one letter and nothing else.\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

Model loaded.


## 3. Model Training

In [8]:
!pip install -q transformers peft accelerate datasets

In [54]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

# 老师要求的模型
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# 设备
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 加载 processor
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 有些模型没有 pad token，这里补一下
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# 加载 model
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

# CPU 情况下手动放到 device
if not torch.cuda.is_available():
    model.to(device)

# 切到推理模式
model.eval()

print("Processor loaded.")
print("Model loaded.")
print("Using device:", model.device)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Processor loaded.
Model loaded.
Using device: cuda:0


In [55]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,080,768 || all params: 509,563,072 || trainable%: 0.4083


In [56]:
# from torch.utils.data import Dataset
# from PIL import Image

# class VQATrainDataset(Dataset):
#     def __init__(self, df, processor, data_dir, img_size):
#         self.df = df.reset_index(drop=True)
#         self.processor = processor
#         self.data_dir = data_dir
#         self.img_size = img_size

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]

#         image = Image.open(self.data_dir / row["image_path"]).convert("RGB").resize((self.img_size, self.img_size))
#         text = build_prompt(row, include_answer=True)

#         enc = self.processor(
#             text=[text],
#             images=[image],
#             return_tensors="pt",
#             padding=False,
#             truncation=False,
#         )

#         item = {}
#         for k, v in enc.items():
#             item[k] = v.squeeze(0)

#         item["labels"] = item["input_ids"].clone()
#         return item

from torch.utils.data import Dataset
from PIL import Image
import torch

class VQATrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, img_size):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(self.data_dir / row["image_path"]).convert("RGB").resize((self.img_size, self.img_size))

        prefix_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        enc = self.processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        prefix_enc = self.processor(
            text=[prefix_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        item = {}
        for k, v in enc.items():
            item[k] = v.squeeze(0)

        labels = item["input_ids"].clone()

        prefix_len = prefix_enc["input_ids"].shape[1]

        # 前面的 prompt 部分不参与 loss
        labels[:prefix_len] = -100

        item["labels"] = labels
        return item

In [57]:
import torch

def multimodal_collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    pixel_values = [x["pixel_values"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id,
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask,
        batch_first=True,
        padding_value=0,
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100,
    )

    pixel_values = torch.stack(pixel_values, dim=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": pixel_values,
        "labels": labels,
    }

In [58]:
# train_small = train_df.sample(min(1000, len(train_df)), random_state=42).reset_index(drop=True)
# val_small = val_df.sample(min(200, len(val_df)), random_state=42).reset_index(drop=True)

train_full = train_df.reset_index(drop=True)
train_dataset = VQATrainDataset(train_full, processor, DATA_DIR, IMG_SIZE)


In [59]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./smolvlm_lora_out",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    eval_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=multimodal_collate_fn,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
20,0.634900
40,0.720500
60,0.630100
80,0.502000
100,0.624900
120,0.483900
140,0.486900
160,0.454900
180,0.491600
200,0.486700


TrainOutput(global_step=778, training_loss=0.38922087392341204, metrics={'train_runtime': 7165.075, 'train_samples_per_second': 0.868, 'train_steps_per_second': 0.109, 'total_flos': 2.391501150897792e+16, 'train_loss': 0.38922087392341204, 'epoch': 2.0})

In [60]:
trainer.save_model("./smolvlm_lora_out/final")
processor.save_pretrained("./smolvlm_lora_out/final")
print("saved.")

saved.


In [61]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/final_dl/my_trained_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))

print("Saved to:", SAVE_DIR)

Saved to: /content/drive/MyDrive/final_dl/my_trained_model6


In [64]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel
import torch

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
SAVE_DIR = "/content/drive/MyDrive/final_dl/my_trained_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(SAVE_DIR)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, SAVE_DIR)

if not torch.cuda.is_available():
    model.to(device)

model.eval()

print("Loaded trained model from:", SAVE_DIR)
print("Using device:", model.device)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded trained model from: /content/drive/MyDrive/final_dl/my_trained_model6
Using device: cuda:0


In [ ]:
# 在val集上验证

from tqdm.auto import tqdm
import pandas as pd
from PIL import Image
import torch

def extract_answer_index(output_text, num_choices):
    text = str(output_text).strip().upper()
    for ch in text:
        if ch in CHOICE_LETTERS[:num_choices]:
            return CHOICE_LETTERS.index(ch)
    return 0

val_preds = []
correct = 0

# for i in tqdm(range(min(100, len(val_df))), desc="Evaluating trained model on first 100 val"):
for i in tqdm(range(len(val_df)), desc="Evaluating trained model on first 100 val"):
    row = val_df.iloc[i]

    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    new_tokens = generated_ids[:, input_len:]
    decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

    pred_idx = extract_answer_index(decoded, len(row["choices"]))
    gt = int(row["answer"])

    val_preds.append({
        "id": row["id"],
        "pred": pred_idx,
        "gt": gt,
        "raw_output": decoded,
        "correct": int(pred_idx == gt),
    })

    if pred_idx == gt:
        correct += 1

val_result_df = pd.DataFrame(val_preds)
acc = correct / len(val_result_df)

print("Validation accuracy on first 100 samples:", acc)
print(val_result_df.head())
print(val_result_df["pred"].value_counts().sort_index())

In [65]:
# 在test集上生成submission file

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

CHOICE_LETTERS = "ABCDEFGHIJ"

def extract_answer_index(output_text, num_choices):
    text = str(output_text).strip().upper()
    for ch in text:
        if ch in CHOICE_LETTERS[:num_choices]:
            return CHOICE_LETTERS.index(ch)
    return 0

pred_rows = []

for i in tqdm(range(len(test_df)), desc="Generating submission with trained model"):
    row = test_df.iloc[i]

    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    new_tokens = generated_ids[:, input_len:]
    decoded = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    pred_idx = extract_answer_index(decoded, len(row["choices"]))

    pred_rows.append({
        "id": row["id"],
        "answer": int(pred_idx),
    })

submission_df = pd.DataFrame(pred_rows)
submission_df.to_csv("submission.csv", index=False)

print(submission_df.head())
print(submission_df.shape)

Generating submission with trained model:   0%|          | 0/1008 [00:00<?, ?it/s]

           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       1
(1008, 2)


In [ ]:
import shutil

SAVE_SUB_PATH = "/content/drive/MyDrive/final_dl/submission.csv"
shutil.copy("submission.csv", SAVE_SUB_PATH)

print("Saved submission to:", SAVE_SUB_PATH)